In [1]:
# import des libs pyspark

from pyspark.sql import SparkSession

In [2]:
# creation d'une session pyspark

spark = SparkSession.builder.appName("MushroomsWkshop").getOrCreate()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 12:23:53 WARN Utils: Your hostname, MacBook-Air-de-Agathe.local, resolves to a loopback address: 127.0.0.1; using 192.168.20.19 instead (on interface en0)
26/10/27 12:23:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/Licas/Desktop/github/pyspark-workshops/venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/27 12:24:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [4]:
# on charge le csv 

df_pyspark = spark.read.csv("mushrooms.csv",inferSchema=True, header=True)

In [5]:
df_pyspark.show(5)

+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|class|cap-shape|cap-surface|cap-color|bruises|odor|gill-attachment|gill-spacing|gill-size|gill-color|stalk-shape|stalk-root|stalk-surface-above-ring|stalk-surface-below-ring|stalk-color-above-ring|stalk-color-below-ring|veil-type|veil-color|ring-number|ring-type|spore-print-color|population|habitat|
+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|    p|        x|          s|        n|      t|   p|              f|           c|        n|   

In [6]:
# focus sur la structure des données et du df

df_pyspark.printSchema()

root
 |-- class: string (nullable = true)
 |-- cap-shape: string (nullable = true)
 |-- cap-surface: string (nullable = true)
 |-- cap-color: string (nullable = true)
 |-- bruises: string (nullable = true)
 |-- odor: string (nullable = true)
 |-- gill-attachment: string (nullable = true)
 |-- gill-spacing: string (nullable = true)
 |-- gill-size: string (nullable = true)
 |-- gill-color: string (nullable = true)
 |-- stalk-shape: string (nullable = true)
 |-- stalk-root: string (nullable = true)
 |-- stalk-surface-above-ring: string (nullable = true)
 |-- stalk-surface-below-ring: string (nullable = true)
 |-- stalk-color-above-ring: string (nullable = true)
 |-- stalk-color-below-ring: string (nullable = true)
 |-- veil-type: string (nullable = true)
 |-- veil-color: string (nullable = true)
 |-- ring-number: string (nullable = true)
 |-- ring-type: string (nullable = true)
 |-- spore-print-color: string (nullable = true)
 |-- population: string (nullable = true)
 |-- habitat: string 

In [7]:
# nous n'avons que des données au format str, il nous faut les convertir en int

from pyspark.ml.feature import StringIndexer

In [8]:
colonnes_cat = df_pyspark.columns
print(colonnes_cat)

['class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']


In [9]:
# On boucle sur chaque nom de colonne individuellement
for c in colonnes_cat:
    # 1. Créer l'indexeur pour la colonne courante 'c'
    indexer = StringIndexer(inputCol=c, outputCol=c + "_encoded")
    
    # 2. Ajuster (fit) et transformer (transform) directement le DataFrame
    df_pyspark = indexer.fit(df_pyspark).transform(df_pyspark)
    
    # 3. (Optionnel) Convertir en entier si besoin
    df_pyspark = df_pyspark.withColumn(c + "_encoded", df_pyspark[c + "_encoded"].cast("int"))

In [10]:
df_pyspark.show(5)

+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+-------------+-----------------+-------------------+-----------------+---------------+------------+-----------------------+--------------------+-----------------+------------------+-------------------+------------------+--------------------------------+--------------------------------+------------------------------+------------------------------+-----------------+------------------+-------------------+-----------------+-------------------------+------------------+---------------+
|class|cap-shape|cap-surface|cap-color|bruises|odor|gill-attachment|gill-spacing|gill-size|gill-color|stalk-shape|stalk-root|stalk-surface-above-ring|stalk-surface-below-ring|stalk-color-above-ring|stalk-color-belo

26/10/27 12:24:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [11]:
len(df_pyspark.columns)

46

In [12]:
encoded_cols = df_pyspark.columns[-23:]
encoded_df = df_pyspark.select(encoded_cols)

In [13]:
encoded_df.show(5)
len(encoded_df.columns)

+-------------+-----------------+-------------------+-----------------+---------------+------------+-----------------------+--------------------+-----------------+------------------+-------------------+------------------+--------------------------------+--------------------------------+------------------------------+------------------------------+-----------------+------------------+-------------------+-----------------+-------------------------+------------------+---------------+
|class_encoded|cap-shape_encoded|cap-surface_encoded|cap-color_encoded|bruises_encoded|odor_encoded|gill-attachment_encoded|gill-spacing_encoded|gill-size_encoded|gill-color_encoded|stalk-shape_encoded|stalk-root_encoded|stalk-surface-above-ring_encoded|stalk-surface-below-ring_encoded|stalk-color-above-ring_encoded|stalk-color-below-ring_encoded|veil-type_encoded|veil-color_encoded|ring-number_encoded|ring-type_encoded|spore-print-color_encoded|population_encoded|habitat_encoded|
+-------------+-------------

23

In [14]:
# data ok, on commence l'extraction des features

from pyspark.ml.feature import VectorAssembler

# on sélectionne uniquement les colonnes correspondants aux features (tout sauf col 1)
feature_columns = encoded_cols[1:]
features = encoded_df.select(feature_columns)

len(features.columns)
features.show(5)


+-----------------+-------------------+-----------------+---------------+------------+-----------------------+--------------------+-----------------+------------------+-------------------+------------------+--------------------------------+--------------------------------+------------------------------+------------------------------+-----------------+------------------+-------------------+-----------------+-------------------------+------------------+---------------+
|cap-shape_encoded|cap-surface_encoded|cap-color_encoded|bruises_encoded|odor_encoded|gill-attachment_encoded|gill-spacing_encoded|gill-size_encoded|gill-color_encoded|stalk-shape_encoded|stalk-root_encoded|stalk-surface-above-ring_encoded|stalk-surface-below-ring_encoded|stalk-color-above-ring_encoded|stalk-color-below-ring_encoded|veil-type_encoded|veil-color_encoded|ring-number_encoded|ring-type_encoded|spore-print-color_encoded|population_encoded|habitat_encoded|
+-----------------+-------------------+-----------------

In [15]:
featureAssembler = VectorAssembler(inputCols= feature_columns ,outputCol="features")
output = featureAssembler.transform(encoded_df)

In [16]:
output.show(5)

+-------------+-----------------+-------------------+-----------------+---------------+------------+-----------------------+--------------------+-----------------+------------------+-------------------+------------------+--------------------------------+--------------------------------+------------------------------+------------------------------+-----------------+------------------+-------------------+-----------------+-------------------------+------------------+---------------+--------------------+
|class_encoded|cap-shape_encoded|cap-surface_encoded|cap-color_encoded|bruises_encoded|odor_encoded|gill-attachment_encoded|gill-spacing_encoded|gill-size_encoded|gill-color_encoded|stalk-shape_encoded|stalk-root_encoded|stalk-surface-above-ring_encoded|stalk-surface-below-ring_encoded|stalk-color-above-ring_encoded|stalk-color-below-ring_encoded|veil-type_encoded|veil-color_encoded|ring-number_encoded|ring-type_encoded|spore-print-color_encoded|population_encoded|habitat_encoded|        

In [17]:
output.select("features","cap-shape_encoded").show(5)

+--------------------+-----------------+
|            features|cap-shape_encoded|
+--------------------+-----------------+
|(22,[1,3,4,7,8,9,...|                0|
|(22,[1,2,3,4,8,9,...|                0|
|(22,[0,1,2,3,4,8,...|                3|
|(22,[2,3,4,7,8,9,...|                0|
|(22,[1,2,6,8,10,1...|                0|
+--------------------+-----------------+
only showing top 5 rows


In [18]:
# on passe au train/test split

model_data = output.select("features", "class_encoded")
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)

print(f"Size of train data is: {train_data.count()}, size of test data is: {test_data.count()}")

Size of train data is: 6573, size of test data is: 1551


# Régression Logistique

In [19]:
# on importe le modèle de régression logistique

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol = 'features', labelCol = 'class_encoded', maxIter=50)

lrModel = lr.fit(train_data)

In [20]:
lr_pred = lrModel.transform(test_data)

In [21]:
lr_pred.show(5)

+--------------------+-------------+--------------------+--------------------+----------+
|            features|class_encoded|       rawPrediction|         probability|prediction|
+--------------------+-------------+--------------------+--------------------+----------+
|(22,[0,1,2,3,4,6,...|            0|[5.22465579503168...|[0.99464660090475...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|[1.06225570493160...|[0.74312137740911...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|[1.06362977426424...|[0.74338358860944...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|[3.93789323485752...|[0.98088333827498...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|[-0.2231327859099...|[0.44444710257055...|       1.0|
+--------------------+-------------+--------------------+--------------------+----------+
only showing top 5 rows


In [22]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="class_encoded",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
print(f"Test Area Under ROC: {evaluator.evaluate(lr_pred)}")

Test Area Under ROC: 0.9976650978696514


In [23]:
lr_pred.groupBy("class_encoded", "prediction").count().show()

+-------------+----------+-----+
|class_encoded|prediction|count|
+-------------+----------+-----+
|            1|       0.0|   11|
|            0|       0.0|  773|
|            1|       1.0|  755|
|            0|       1.0|   12|
+-------------+----------+-----+



# Arbre de décision

In [24]:
# on teste avec Arbre de décision

from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# training
dt = DecisionTreeClassifier(featuresCol = 'features', labelCol = 'class_encoded', maxDepth = 3)

dtModel = dt.fit(train_data)


In [25]:
# test

dt_pred = dtModel.transform(test_data)

In [26]:
dt_pred.show(5)

+--------------------+-------------+-------------+--------------------+----------+
|            features|class_encoded|rawPrediction|         probability|prediction|
+--------------------+-------------+-------------+--------------------+----------+
|(22,[0,1,2,3,4,6,...|            0|  [71.0,69.0]|[0.50714285714285...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|  [71.0,69.0]|[0.50714285714285...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|  [71.0,69.0]|[0.50714285714285...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|  [71.0,69.0]|[0.50714285714285...|       0.0|
|(22,[0,1,2,3,4,6,...|            0|  [71.0,69.0]|[0.50714285714285...|       0.0|
+--------------------+-------------+-------------+--------------------+----------+
only showing top 5 rows


In [27]:
# on évalue les perfs de ce modèle

evaluator = BinaryClassificationEvaluator(
    labelCol="class_encoded",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
print(f"Test Area Under ROC: {evaluator.evaluate(dt_pred)}")

Test Area Under ROC: 0.9610675026192813


# Random Forest

In [28]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [29]:
#Training Model
rf = RandomForestClassifier(featuresCol = 'features', labelCol = 'class_encoded', numTrees = 500, maxDepth = 10)
rfModel = rf.fit(train_data)

26/10/27 12:24:21 WARN DAGScheduler: Broadcasting large task binary with size 1052.1 KiB
26/10/27 12:24:21 WARN DAGScheduler: Broadcasting large task binary with size 1445.6 KiB
26/10/27 12:24:22 WARN DAGScheduler: Broadcasting large task binary with size 1748.8 KiB
26/10/27 12:24:23 WARN DAGScheduler: Broadcasting large task binary with size 1856.0 KiB
26/10/27 12:24:23 WARN DAGScheduler: Broadcasting large task binary with size 1711.2 KiB
26/10/27 12:24:23 WARN DAGScheduler: Broadcasting large task binary with size 1287.8 KiB


In [30]:
rf_pred = rfModel.transform(test_data)

In [31]:
evaluator = BinaryClassificationEvaluator(
    labelCol="class_encoded",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
print(f"Test Area Under ROC with RF: {evaluator.evaluate(rf_pred)}")

26/10/27 12:24:25 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB


Test Area Under ROC with RF: 1.0


# Synthèse comparative

In [32]:
import pandas as pd

def summarize(name, pred):
    auc = evaluator.evaluate(pred)
    tp = pred.filter("class_encoded = 1 AND prediction = 1").count()
    tn = pred.filter("class_encoded = 0 AND prediction = 0").count()
    fp = pred.filter("class_encoded = 0 AND prediction = 1").count()
    fn = pred.filter("class_encoded = 1 AND prediction = 0").count()
    total = tp + tn + fp + fn
    return {
        "Modèle": name,
        "AUC": round(auc, 4),
        "Accuracy": round((tp + tn) / total, 4),
        "Faux négatifs (vénéneux ratés)": fn,
        "Faux positifs": fp,
    }

synthese = pd.DataFrame([
    summarize("Régression logistique", lr_pred),
    summarize("Arbre de décision", dt_pred),
    summarize("Random Forest", rf_pred),
]).set_index("Modèle")

synthese

26/10/27 12:24:28 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/10/27 12:24:28 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/10/27 12:24:29 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/10/27 12:24:29 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/10/27 12:24:30 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB


,AUC,Accuracy,Faux négatifs (vénéneux ratés),Faux positifs
Modèle,,,,
Régression logistique,0.9977,0.9852,11,12
Arbre de décision,0.9611,0.9749,35,4
Random Forest,1.0000,1.0000,0,0
